# 02 - Upload source files to S3
Uploads every file **exactly as received**: no filtering, no cleaning, no dropped rows (bad records are caught later by the validation step).

Landing layout:
```
members/ingest_date=YYYYMMDD/SkyPoints_Members_YYYYMMDD.txt
redemptions/ingest_date=YYYYMMDD/SkyPoints_Redemptions_YYYYMMDD.json
reference/<COUNTRY>.xlsx            (USA / IND / AUS sample files)
```

In [ ]:
import re
import boto3
from pathlib import Path

In [ ]:
# ---- CONFIG: edit these two values ----
SUFFIX = "yourname123"          # must make bucket names globally unique
REGION = "ap-southeast-7"       # Thailand (opt-in region: must be enabled in your AWS account)
PROFILE = None                  # AWS named profile, or None for default credentials

PREFIX = "skypoints"
BUCKETS = {
    "landing": f"{PREFIX}-landing-{SUFFIX}",   # files Snowflake COPY INTO reads
    "archive": f"{PREFIX}-archive-{SUFFIX}",   # files moved here after a successful load
    "rejects": f"{PREFIX}-rejects-{SUFFIX}",   # rejected-record extracts from COPY INTO
}
BUCKETS

DATA_DIR = Path("sample_data")   # folder holding the files to upload

In [ ]:
session = boto3.Session(profile_name=PROFILE, region_name=REGION)
s3 = session.client("s3", region_name=REGION)

FEEDS = {
    "members":     re.compile(r"^SkyPoints_Members_(\d{8})\.txt$"),
    "redemptions": re.compile(r"^SkyPoints_Redemptions_(\d{8})\.json$"),
}

plan = []   # (local path, bucket, key)
for f in sorted(DATA_DIR.iterdir()):
    if f.is_file():
        for feed, pattern in FEEDS.items():
            m = pattern.match(f.name)
            if m:
                plan.append((f, BUCKETS["landing"], f"{feed}/ingest_date={m.group(1)}/{f.name}"))
                break
        else:
            print("WARNING not a known feed, not uploaded:", f.name)

for f in sorted((DATA_DIR / "reference").glob("*.xlsx")):
    plan.append((f, BUCKETS["landing"], f"reference/{f.name}"))

for f, b, k in plan:
    print(f"{f}  ->  s3://{b}/{k}")

In [ ]:
# upload (byte-for-byte)
for f, bucket, key in plan:
    s3.upload_file(str(f), bucket, key)
    print("uploaded", key)

In [ ]:
# verify nothing was lost: size must match, and for text feeds the line count must match too
def line_count(data: bytes) -> int:
    return len(data.splitlines())

ok = True
for f, bucket, key in plan:
    head = s3.head_object(Bucket=bucket, Key=key)
    size_ok = head["ContentLength"] == f.stat().st_size
    msg = f"{key}: size local={f.stat().st_size} s3={head['ContentLength']}"
    if f.suffix in (".txt", ".json"):
        local_lines = line_count(f.read_bytes())
        s3_lines = line_count(s3.get_object(Bucket=bucket, Key=key)["Body"].read())
        size_ok = size_ok and local_lines == s3_lines
        msg += f", lines local={local_lines} s3={s3_lines}"
    print(("OK    " if size_ok else "MISMATCH ") + msg)
    ok &= size_ok
assert ok, "Upload verification failed"
print("All files uploaded and verified.")